In [0]:

# Project configuration
base_path = "/Volumes/adv_dev_westus3_demo_001/ecommerce/lakehouse_vol"

catalog_name = "adv_dev_westus3_demo_001"
schema_name = "ecommerce"

silver_schema = f"{catalog_name}.{schema_name}"
gold_schema = f"{catalog_name}.{schema_name}"

print("Silver schema:", silver_schema)
print("Gold schema:", gold_schema)


Silver schema: adv_dev_westus3_demo_001.ecommerce
Gold schema: adv_dev_westus3_demo_001.ecommerce


In [0]:

# Choose which Gold table to build
dbutils.widgets.text("gold_table", "ALL", "Gold table name")

gold_table = dbutils.widgets.get("gold_table").strip().upper()

print("Selected Gold table:", gold_table)


Selected Gold table: ALL


In [0]:

from pyspark.sql.functions import col

# Define Gold tables and their source tables
gold_tables = {
    "DIM_CUSTOMERS": "silver_customers",
    "DIM_PRODUCTS": "silver_products",
    "FACT_ORDER_SALES": "silver_order_items"
}

# Select requested table(s)
if gold_table == "ALL":
    selected_tables = list(gold_tables.keys())
elif gold_table in gold_tables:
    selected_tables = [gold_table]
else:
    raise ValueError(
        "Use ALL, DIM_CUSTOMERS, DIM_PRODUCTS, or FACT_ORDER_SALES"
    )

# Build customer and product dimensions
for table in selected_tables:
    if table == "DIM_CUSTOMERS":
        df = spark.table(f"{silver_schema}.silver_customers")
        df = df.dropDuplicates(["customer_id"])
        target_table = "gold_dim_customers"

    elif table == "DIM_PRODUCTS":
        df = spark.table(f"{silver_schema}.silver_products")
        df = df.dropDuplicates(["product_id"])
        target_table = "gold_dim_products"

    elif table == "FACT_ORDER_SALES":
        print("Sales fact table will be created in the next step.")
        continue

    df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"{gold_schema}.{target_table}")

    print(f"Created {target_table} successfully.")

print("Gold dimension processing completed.")


Created gold_dim_customers successfully.
Created gold_dim_products successfully.
Sales fact table will be created in the next step.
Gold dimension processing completed.


In [0]:

from pyspark.sql.functions import sum as spark_sum

if gold_table in ["ALL", "FACT_ORDER_SALES"]:

    # Read Silver tables
    orders = spark.table(f"{silver_schema}.silver_orders")
    items = spark.table(f"{silver_schema}.silver_order_items")
    payments = spark.table(f"{silver_schema}.silver_order_payments")

    # Aggregate payments to one row per order
    payments_agg = payments.groupBy("order_id").agg(
        spark_sum("payment_value").alias("total_payment_value")
    )

    # Join orders with order items
    fact_df = items.join(
        orders.select(
            "order_id",
            "customer_id",
            "order_status",
            "order_purchase_timestamp"
        ),
        on="order_id",
        how="left"
    )

    # Join aggregated payments without multiplying order-item rows
    fact_df = fact_df.join(
        payments_agg,
        on="order_id",
        how="left"
    )

    # Save the Gold fact table
    fact_df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"{gold_schema}.gold_fact_order_sales")

    print("Created gold_fact_order_sales successfully.")

else:
    print("Skipping fact table because it was not selected.")


Created gold_fact_order_sales successfully.


In [0]:

# Verify Gold tables and their row counts
gold_tables_to_check = [
    "gold_dim_customers",
    "gold_dim_products",
    "gold_fact_order_sales"
]

for name in gold_tables_to_check:
    full_name = f"{gold_schema}.{name}"
    print(f"{name}: {spark.table(full_name).count()} rows")


gold_dim_customers: 99441 rows
gold_dim_products: 32951 rows
gold_fact_order_sales: 112650 rows


In [0]:

from pyspark.sql.functions import col

# Load Gold tables
customers = spark.table(f"{gold_schema}.gold_dim_customers")
products = spark.table(f"{gold_schema}.gold_dim_products")
sales = spark.table(f"{gold_schema}.gold_fact_order_sales")

# Check missing customer IDs
missing_customer_ids = customers.filter(
    col("customer_id").isNull()
).count()

# Check missing product IDs
missing_product_ids = products.filter(
    col("product_id").isNull()
).count()

# Check missing order IDs in sales
missing_order_ids = sales.filter(
    col("order_id").isNull()
).count()

print("Missing customer IDs:", missing_customer_ids)
print("Missing product IDs:", missing_product_ids)
print("Missing order IDs in sales:", missing_order_ids)


Missing customer IDs: 0
Missing product IDs: 0
Missing order IDs in sales: 0


In [0]:

# Reusable data quality function
def check_data_quality(df, key_column):
    total_rows = df.count()

    missing_keys = df.filter(
        col(key_column).isNull()
    ).count()

    duplicate_keys = (
        df.groupBy(key_column)
        .count()
        .filter(col("count") > 1)
        .count()
    )

    return {
        "total_rows": total_rows,
        "missing_keys": missing_keys,
        "duplicate_keys": duplicate_keys
    }

print("Data quality function created successfully.")


Data quality function created successfully.


In [0]:

from pyspark.sql.functions import col

# Define the schema
catalog_name = "adv_dev_westus3_demo_001"
schema_name = "ecommerce"
gold_schema = f"{catalog_name}.{schema_name}"

# Define tables and their business keys
quality_checks = {
    "gold_dim_customers": ["customer_id"],
    "gold_dim_products": ["product_id"],
    "gold_fact_order_sales": ["order_id", "order_item_id"]
}

# Run data quality checks dynamically
for table_name, key_columns in quality_checks.items():
    df = spark.table(f"{gold_schema}.{table_name}")

    total_rows = df.count()

    missing_keys = 0
    for key_column in key_columns:
        missing_keys += df.filter(col(key_column).isNull()).count()

    duplicate_keys = (
        df.groupBy(*key_columns)
          .count()
          .filter(col("count") > 1)
          .count()
    )

    print(f"\nTable: {table_name}")
    print("Total rows:", total_rows)
    print("Missing key values:", missing_keys)
    print("Duplicate key combinations:", duplicate_keys)

    # Fail the pipeline if any validation fails
    if total_rows == 0:
        raise ValueError(f"FAILED: {table_name} is empty.")

    if missing_keys > 0:
        raise ValueError(f"FAILED: Missing keys in {table_name}.")

    if duplicate_keys > 0:
        raise ValueError(f"FAILED: Duplicate keys in {table_name}.")

print("\nAll data quality checks passed.")





Table: gold_dim_customers
Total rows: 99441
Missing key values: 0
Duplicate key combinations: 0

Table: gold_dim_products
Total rows: 32951
Missing key values: 0
Duplicate key combinations: 0

Table: gold_fact_order_sales
Total rows: 112650
Missing key values: 0
Duplicate key combinations: 0

All data quality checks passed.


In [0]:
%sql
DESCRIBE HISTORY adv_dev_westus3_demo_001.ecommerce.gold_dim_customers

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-10-23T06:42:43.407Z,143222325238660,rajnandinikuberkar9763@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""databricks.internal.autoUpgrades.delta.enableRowTracking"":""true"",""delta.enableDeletionVectors"":""true"",""databricks.internal.autoUpgrades.delta.checkpointPolicy"":""v2"",""databricks.internal.autoUpgrades.delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""050413ac-d70f-4cb0-815c-4b8663f2d508"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-f014124a-929d-4886-ba20-043e106b4286"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-d82237e3-3416-4e9b-9808-8f362f305c95"",""databricks.internal.autoUpgrades.delta.feature.catalogManaged"":""supported"",""delta.enableInCommitTimestamps"":""true""}, statsOnLoad -> true, clusterByAuto -> true, clusteringOnWriteStatus -> Reason for skipping: Nonexistent clustering column, generally when replacing a Liquid table with a table without clustering columns)",null,List(1457515604045337),7ae6bd2c-0106-4676-b173-905dc915f468,1023-063014-yex4xjum-v2n,1,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 4062561, numDeletionVectorsRemoved -> 0, numOutputRows -> 99441, numOutputBytes -> 4062561)",null,Databricks-Runtime/19.12.x-photon-scala2.13
1,2026-10-23T06:35:26.871Z,143222325238660,rajnandinikuberkar9763@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""databricks.internal.autoUpgrades.delta.enableRowTracking"":""true"",""delta.enableDeletionVectors"":""true"",""databricks.internal.autoUpgrades.delta.checkpointPolicy"":""v2"",""databricks.internal.autoUpgrades.delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""050413ac-d70f-4cb0-815c-4b8663f2d508"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-f014124a-929d-4886-ba20-043e106b4286"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-d82237e3-3416-4e9b-9808-8f362f305c95"",""databricks.internal.autoUpgrades.delta.feature.catalogManaged"":""supported"",""delta.enableInCommitTimestamps"":""true""}, statsOnLoad -> true, clusterByAuto -> true, clusteringOnWriteStatus -> Reason for skipping: Nonexistent clustering column, generally when replacing a Liquid table with a table without clustering columns)",null,List(1457515604045337),ab329d6c-8abe-494f-9773-644cbf76224e,1023-063014-yex4xjum-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 4062561, numDeletionVectorsRemoved -> 0, numOutputRows -> 99441, numOutputBytes -> 4062561)",null,Databricks-Runtime/19.12.x-photon-scala2.13
0,2026-10-22T10:50:12.094Z,143222325238660,rajnandinikuberkar9763@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""databricks.internal.autoUpgrades.delta.enableRowTracking"":""true"",""delta.enableDeletionVectors"":""true"",""databricks.internal.autoUpgrades.delta.checkpointPolicy"":""v2"",""databricks.internal.autoUpgrades.delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRow

In [0]:
%sql
SELECT *
FROM adv_dev_westus3_demo_001.ecommerce.gold_dim_customers
VERSION AS OF 0
LIMIT 10;

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
c6e04ca05bff626899b98ac1e68a03d1,11cb374299fc168cdb79345ca63371d1,80420,curitiba,PR
e0753c85c87c807e556152043b16bed7,5defeaf938551c166918880cb5d23c5c,87528,alto paraiso,PR
76d4a3d0005fc2a1c3d995a062bfc72d,1fdb1b2757402b1c8c077a9e197b5d63,9070,santo andre,SP
de956ad243196560aa9e66ca05681f65,af4e6f96718efb093cd3c91428bcab10,4053,sao paulo,SP
eace6dd8a0f090e33030802dece9184d,85ea73a13d42ec297a014ed14914a555,3191,sao paulo,SP
c96b4cfef71ae1ab097480c18abc21e2,59c4294186b35d084e67f0112c7a1449,85803,cascavel,PR
c4ebc72b14f3ffdea1626db488208713,f2092562581c6e5e6d808e78e78ab862,14027,ribeirao preto,SP
f915ff705480d8952c4e91371c39e95e,5f55921b0fe67e51a904be6a9853e632,60192,fortaleza,CE
55acee103cfe584bc279a010f6c6dab4,70d58870c1a5f29a875f77716e5b34c6,42840,abrantes,BA
0a6851c4607181b4a4a770cbe556b822,82809cee3c35eb89b4dd4d2d1a22eedf,13040,campinas,SP


In [0]:
%sql
SELECT *
FROM adv_dev_westus3_demo_001.ecommerce.gold_dim_customers
VERSION AS OF 0
LIMIT 10;

current_count,version_0_count
99441,99441


In [0]:
%sql
DESCRIBE HISTORY adv_dev_westus3_demo_001.ecommerce.gold_dim_products

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-10-23T06:42:45.765Z,143222325238660,rajnandinikuberkar9763@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""databricks.internal.autoUpgrades.delta.enableRowTracking"":""true"",""delta.enableDeletionVectors"":""true"",""databricks.internal.autoUpgrades.delta.checkpointPolicy"":""v2"",""databricks.internal.autoUpgrades.delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""f258b5d4-0fca-46f7-8fa6-c34999a35d30"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-b6e36f85-9601-4ac0-b374-97de6793d7a3"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-1f9af49f-88db-445d-9081-ab5be31bd58a"",""databricks.internal.autoUpgrades.delta.feature.catalogManaged"":""supported"",""delta.enableInCommitTimestamps"":""true""}, statsOnLoad -> true, clusterByAuto -> true, clusteringOnWriteStatus -> Reason for skipping: Nonexistent clustering column, generally when replacing a Liquid table with a table without clustering columns)",null,List(1457515604045337),b79ce176-f6c4-414e-9b26-8a415fcf3553,1023-063014-yex4xjum-v2n,1,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 862642, numDeletionVectorsRemoved -> 0, numOutputRows -> 32951, numOutputBytes -> 862642)",null,Databricks-Runtime/19.12.x-photon-scala2.13
1,2026-10-23T06:35:31.081Z,143222325238660,rajnandinikuberkar9763@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""databricks.internal.autoUpgrades.delta.enableRowTracking"":""true"",""delta.enableDeletionVectors"":""true"",""databricks.internal.autoUpgrades.delta.checkpointPolicy"":""v2"",""databricks.internal.autoUpgrades.delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""f258b5d4-0fca-46f7-8fa6-c34999a35d30"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-b6e36f85-9601-4ac0-b374-97de6793d7a3"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-1f9af49f-88db-445d-9081-ab5be31bd58a"",""databricks.internal.autoUpgrades.delta.feature.catalogManaged"":""supported"",""delta.enableInCommitTimestamps"":""true""}, statsOnLoad -> true, clusterByAuto -> true, clusteringOnWriteStatus -> Reason for skipping: Nonexistent clustering column, generally when replacing a Liquid table with a table without clustering columns)",null,List(1457515604045337),817c9d1e-c3db-4ec7-95f2-13c9893c9117,1023-063014-yex4xjum-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 862642, numDeletionVectorsRemoved -> 0, numOutputRows -> 32951, numOutputBytes -> 862642)",null,Databricks-Runtime/19.12.x-photon-scala2.13
0,2026-10-22T10:50:15.113Z,143222325238660,rajnandinikuberkar9763@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""databricks.internal.autoUpgrades.delta.enableRowTracking"":""true"",""delta.enableDeletionVectors"":""true"",""databricks.internal.autoUpgrades.delta.checkpointPolicy"":""v2"",""databricks.internal.autoUpgrades.delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTrac

In [0]:
%sql
SELECT *
FROM adv_dev_westus3_demo_001.ecommerce.gold_dim_products
VERSION AS OF 0
LIMIT 10;

product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
e1d1d22e9f8122a4ec1533b032c12562,ferramentas_jardim,49,1163,9,2150,70,8,34
ce5b91848b91118daffb3af53b747475,esporte_lazer,50,699,4,1388,34,9,31
1c6fb703c624b381a20f21f757694866,brinquedos,33,471,2,725,22,17,15
4e04ffb7dd3739ecfc37de8927dd586c,papelaria,59,144,1,500,24,7,16
0992c6cba95a13bfa68ea7d5e22d478b,ferramentas_jardim,42,671,1,3850,29,12,24
32ee3f4e2493045726807808d7abbab6,cama_mesa_banho,54,381,2,1400,38,9,27
af16005fca813272caf59c432153949e,moveis_decoracao,55,1186,2,800,53,8,20
b2b3503e675abbc07e278151da072ed2,moveis_decoracao,46,287,1,8500,75,25,55
acd427ee119d5c71c7a85ae488cb0a6a,fashion_bolsas_e_acessorios,52,68,7,200,16,2,11
51fed4afb1b41d00ad9d25a73b7fbbc7,consoles_games,48,303,3,250,21,7,15


In [0]:
%sql
SELECT
    (SELECT COUNT(*)
     FROM adv_dev_westus3_demo_001.ecommerce.gold_dim_products) AS current_count,
    (SELECT COUNT(*)
     FROM adv_dev_westus3_demo_001.ecommerce.gold_dim_products VERSION AS OF 0) AS version_0_count;

current_count,version_0_count
32951,32951


In [0]:
%sql

CREATE OR REPLACE TABLE adv_dev_westus3_demo_001.ecommerce.rollback_test
USING DELTA
AS
SELECT *
FROM adv_dev_westus3_demo_001.ecommerce.gold_dim_customers;


num_affected_rows,num_inserted_rows


In [0]:
%sql

DESCRIBE HISTORY adv_dev_westus3_demo_001.ecommerce.rollback_test;


version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-10-23T06:46:09.730Z,143222325238660,rajnandinikuberkar9763@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""databricks.internal.autoUpgrades.delta.enableRowTracking"":""true"",""delta.enableDeletionVectors"":""true"",""databricks.internal.autoUpgrades.delta.checkpointPolicy"":""v2"",""databricks.internal.autoUpgrades.delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""64792613-b7d9-4a44-85fd-f77d905edba1"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-57556d73-ee80-429b-abcf-cfdcb7b76452"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-c3a43301-f338-40ac-a9e3-718bfb21e05e"",""databricks.internal.autoUpgrades.delta.feature.catalogManaged"":""supported""}, statsOnLoad -> true, clusterByAuto -> true, clusteringOnWriteStatus -> Reason for skipping: Nonexistent clustering column, generally when replacing a Liquid table with a table without clustering columns)",null,List(1457515604045337),4d000b27-21c2-40f9-aa5f-83c23aa38a4e,1023-063014-yex4xjum-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 99441, numOutputBytes -> 4062561)",null,Databricks-Runtime/19.12.x-photon-scala2.13


In [0]:
%sql

UPDATE adv_dev_westus3_demo_001.ecommerce.rollback_test
SET customer_city = 'incorrect_test_city'
WHERE customer_id = (
    SELECT customer_id
    FROM adv_dev_westus3_demo_001.ecommerce.rollback_test
    LIMIT 1
);


num_affected_rows
1


In [0]:
%sql

RESTORE TABLE adv_dev_westus3_demo_001.ecommerce.rollback_test
TO VERSION AS OF 0;


table_size_after_restore,num_of_files_after_restore,num_removed_files,num_restored_files,removed_files_size,restored_files_size
4062561,1,2,1,4065432,4062561


In [0]:

from pyspark.sql.functions import col, to_date, current_date

catalog_name = "adv_dev_westus3_demo_001"
schema_name = "ecommerce"
target_schema = f"{catalog_name}.{schema_name}"

orders = spark.table(f"{target_schema}.silver_orders").alias("o")
items = spark.table(f"{target_schema}.silver_order_items").alias("i")
customers = spark.table(f"{target_schema}.silver_customers").alias("c")
products = spark.table(f"{target_schema}.silver_products").alias("p")

fact_sales = (
    items
    .join(orders, col("i.order_id") == col("o.order_id"), "inner")
    .join(customers, col("o.customer_id") == col("c.customer_id"), "left")
    .join(products, col("i.product_id") == col("p.product_id"), "left")
    .select(
        col("i.order_id"),
        col("i.order_item_id"),
        col("o.customer_id"),
        col("i.product_id"),
        col("o.order_purchase_timestamp"),
        to_date(col("o.order_purchase_timestamp")).alias("order_date"),
        col("i.price"),
        col("i.freight_value"),
        (col("i.price") + col("i.freight_value")).alias("revenue"),
        col("c.customer_state"),
        col("p.product_category_name")
    )
    .withColumn("load_date", current_date())
)

(
    fact_sales.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{target_schema}.fact_sales")
)

print("fact_sales created successfully.")
print("Total records:", spark.table(f"{target_schema}.fact_sales").count())


fact_sales created successfully.
Total records: 112650


In [0]:

from pyspark.sql.functions import col

df = spark.table(f"{target_schema}.fact_sales")

duplicate_items = (
    df.groupBy("order_id", "order_item_id")
      .count()
      .filter(col("count") > 1)
      .count()
)

null_revenue = df.filter(col("revenue").isNull()).count()
negative_revenue = df.filter(col("revenue") < 0).count()

print("Duplicate order-item combinations:", duplicate_items)
print("Null revenue:", null_revenue)
print("Negative revenue:", negative_revenue)

if duplicate_items == 0 and null_revenue == 0 and negative_revenue == 0:
    print("Fact sales quality checks passed.")
else:
    raise ValueError("Fact sales quality checks failed.")


Duplicate order-item combinations: 0
Null revenue: 0
Negative revenue: 0
Fact sales quality checks passed.


In [0]:

from pyspark.sql.functions import (
    sum as spark_sum, countDistinct, desc,
    to_date, date_format
)

fact_table = f"{target_schema}.fact_sales"
df = spark.table(fact_table)

# 1. Sales by state
sales_by_state = (
    df.groupBy("customer_state")
      .agg(
          spark_sum("revenue").alias("total_revenue"),
          countDistinct("order_id").alias("total_orders")
      )
)

sales_by_state.write.format("delta").mode("overwrite").saveAsTable(
    f"{target_schema}.gold_sales_by_state"
)

# 2. Top products by revenue
top_products = (
    df.groupBy("product_id", "product_category_name")
      .agg(
          spark_sum("revenue").alias("total_revenue")
      )
      .orderBy(desc("total_revenue"))
)

top_products.write.format("delta").mode("overwrite").saveAsTable(
    f"{target_schema}.gold_top_products_by_revenue"
)

# 3. Daily sales trend
daily_sales = (
    df.groupBy("order_date")
      .agg(
          spark_sum("revenue").alias("total_revenue"),
          countDistinct("order_id").alias("total_orders")
      )
)

daily_sales.write.format("delta").mode("overwrite").saveAsTable(
    f"{target_schema}.gold_daily_sales"
)

# 4. Monthly sales trend
monthly_sales = (
    df.withColumn("sales_month", date_format("order_date", "yyyy-MM"))
      .groupBy("sales_month")
      .agg(
          spark_sum("revenue").alias("total_revenue"),
          countDistinct("order_id").alias("total_orders")
      )
)

monthly_sales.write.format("delta").mode("overwrite").saveAsTable(
    f"{target_schema}.gold_monthly_sales"
)

print("Gold aggregation tables created successfully.")
for table_name in [
    "gold_sales_by_state",
    "gold_top_products_by_revenue",
    "gold_daily_sales",
    "gold_monthly_sales"
]:
    full_name = f"{target_schema}.{table_name}"
    print(table_name, "records:", spark.table(full_name).count())


Gold aggregation tables created successfully.
gold_sales_by_state records: 27
gold_top_products_by_revenue records: 32951
gold_daily_sales records: 616
gold_monthly_sales records: 24


In [0]:

for table_name in [
    "gold_sales_by_state",
    "gold_top_products_by_revenue",
    "gold_daily_sales",
    "gold_monthly_sales"
]:
    print(f"\n--- {table_name} ---")
    spark.table(f"{target_schema}.{table_name}").show(5, truncate=False)



--- gold_sales_by_state ---
+--------------+------------------+------------+
|customer_state|total_revenue     |total_orders|
+--------------+------------------+------------+
|BA            |611506.67         |3358        |
|PI            |108132.28000000003|493         |
|MG            |1856161.4899999988|11544       |
|MT            |186168.96000000002|903         |
|CE            |275606.3          |1327        |
+--------------+------------------+------------+
only showing top 5 rows

--- gold_top_products_by_revenue ---
+--------------------------------+----------------------+------------------+
|product_id                      |product_category_name |total_revenue     |
+--------------------------------+----------------------+------------------+
|bb50f2e236e5eea0100680137654686c|beleza_saude          |67606.10000000002 |
|d1c427060a0f73f6b889a5c7c61f2ac4|informatica_acessorios|60976.03000000001 |
|6cdd53843498f92890544667809f1595|beleza_saude          |59093.989999999954|
|99a47